In [ ]:
# 1. Load dataset
import datasets

ds = datasets.load_dataset('hihihohohehe/vifactcheck-normalized', split='train')

# 2. Load summarizes
import json
with open("summaries_train_result.json", "r", encoding="utf-8") as f:
    summarizes = json.load(f)

# 3. Load model
from sentence_transformers import SentenceTransformer

# Tải model từ HuggingFace (nó sẽ tự động tải về máy bạn)
model = SentenceTransformer('BAAI/bge-m3')

# 4.  Chunk context
import re
from sentence_transformers import SentenceTransformer
import json

def chunk_and_embed_context(context_text, max_len=1000):
    """
    Chunk Context text và embed từng chunk
    
    Args:
        context_text: Text từ cột Context
        max_len: Độ dài tối đa của mỗi chunk (ký tự)
    
    Returns:
        List of dict chứa chunk text và embedding
    """
    # Pattern để split câu tiếng Việt
    pattern = r'(?<=[a-zà-ỹ]{3}[.!?])\s+(?=[A-ZÀ-Ỵ""])'
    
    # Split thành các câu
    sentences = re.split(pattern, context_text)
    
    # Chunking logic
    chunks = []
    chunk = ""
    total_len = 0
    
    for sentence in sentences:
        # Nếu thêm câu này vào sẽ vượt max_len
        if total_len + len(sentence) >= max_len and chunk:
            chunks.append(chunk.strip())
            chunk = sentence + " "
            total_len = len(sentence)
        else:
            chunk += (sentence + " ")
            total_len += len(sentence)
    
    # Thêm chunk cuối cùng nếu còn
    if chunk.strip():
        chunks.append(chunk.strip())
    
    return chunks


def process_article(dataset, model, summarizes, article_idx):
    """
    Process 1 article: chunk Context và embed chunks, evidence, summarize
    
    Args:
        dataset: Dataset từ HuggingFace
        model: SentenceTransformer model để embed
        article_idx: Index của article trong dataset
        summarizes: List chứa các summarize
    
    Returns:
        Dict chứa thông tin article và chunks đã embed
    """
    # FIXED: Truy cập đúng cách - lấy từng column theo index
    context = dataset['Context'][article_idx]
    evidence = dataset['Evidence'][article_idx]
    statement = dataset['Statement'][article_idx]
    label = dataset['labels'][article_idx]
    topic = dataset['Merged Topic'][article_idx]
    topic_label = dataset['label_id'][article_idx]
    summarize = summarizes[str(article_idx)]
    
    # Chunk evidence
    chunks = chunk_and_embed_context(context)

    all_infor = [statement, summarize, evidence] + chunks

    # Embed từng chunk
    embeddings = model.encode(all_infor, convert_to_tensor=False)
    
    # Tạo result
    result = {
        'article_idx': article_idx,
        'topic': topic,
        'topic_label': topic_label,
        'label': label,
        'statement': statement,
        'statement_embedding': embeddings[0].tolist(),
        'summarize': summarize,
        'summarize_embedding': embeddings[1].tolist(),
        'evidence': evidence,
        'evidence_embedding': embeddings[2].tolist(),
        'chunks': []
    }
    
    for i, (chunk_text, embedding) in enumerate(zip(chunks, embeddings)):
        result['chunks'].append({
            'chunk_idx': i,
            'text': chunk_text,
            'embedding': embedding.tolist()  # Convert numpy array to list để save JSON
        })
    
    return result

# 5. embedding
import json
import os

def process_and_save_incremental(dataset, model, summarizes, output_file='evidence_chunks_embedded.json', num_articles=200):
    """
    Process 200 articles và save incremental (mỗi article xong là lưu ngay)
    
    Args:
        dataset: Dataset từ HuggingFace
        model: SentenceTransformer model
        output_file: Tên file output JSON
        num_articles: Số lượng articles cần process (default 200)
    """
    results = []
    
    # Nếu file đã tồn tại, load lên để tiếp tục
    if os.path.exists(output_file):
        print(f">> File {output_file} đã tồn tại, đang load...")
        with open(output_file, 'r', encoding='utf-8') as f:
            results = json.load(f)
        print(f">> Đã load {len(results)} articles từ file")
        start_idx = len(results)
    else:
        start_idx = 0
    
    print(f"\n>> Bắt đầu process từ article {start_idx} đến {num_articles}...")
    
    for i in range(start_idx, num_articles):
        try:
            # Process article - FIXED: truyền dataset thay vì dataset[i]
            result = process_article(dataset, model, summarizes, i)
            results.append(result)
            
            # Save ngay sau mỗi article
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            
            # Progress update mỗi 10 articles
            if (i + 1) % 10 == 0:
                print(f"   ✓ Processed and saved {i + 1}/{num_articles} articles")
        
        except Exception as e:
            print(f"   ✗ Error at article {i}: {str(e)}")
            continue
    
    print(f"\n>> DONE! Đã process và save {len(results)} articles vào {output_file}")
    return results

results_200 = process_and_save_incremental(ds, model, summarizes, output_file='evidence_chunks_embedded.json', num_articles=200)

KeyboardInterrupt: 